In [1]:
import spacy
nlp = spacy.load("de_core_news_sm")

In [2]:
import pandas as pd

task1 = pd.read_json("Input/Data/train/test_task1.json")
data = task1[task1["flausch"] == "yes"].copy()

In [101]:
# returns root

def root(token):
    if token.dep_ == "ROOT":
        return token
    else:
        return root(token.head)

In [102]:
import copy

def words_to_indices(comment, word_spans):

    index_spans = copy.deepcopy(word_spans)

    spans_id = 0
    span_id = 0

    for i in range(len(comment) + 1):

        # if last characters in sentence are identical
        # doesn't yet deal with repetition of characters 
        # at the end of a span within the comment
        if spans_id >= len(word_spans):
            index_spans[spans_id - 1][1] = i
            continue

        looking_for = word_spans[spans_id][span_id]

        if span_id == 0:
            if comment[i:].startswith(looking_for):
                index_spans[spans_id][span_id] = i
                # move to looking for end of span
                span_id = 1
        else:
            if comment[:i].endswith(looking_for):
                index_spans[spans_id][span_id] = i
                # move to beginning of next span
                spans_id += 1
                span_id = 0

    return index_spans

In [103]:
def get_spans(comment):

    head = None
    last = None
    spans = []
    current_span = []

    for token in nlp(comment):

        if root(token) != head:

            if last != None: # not needed for first span
                # save end of last span
                current_span.append(last.text)
                spans.append(current_span)

            # start new span
            current_span = [token.text]
            head = root(token)
        
        last = token

    # end span for last token
    current_span.append(last.text)
    spans.append(current_span)

    return words_to_indices(comment, spans)

In [104]:
print(get_spans(data.iloc[0]["comment"]))
print(data.iloc[0]["span_pairs"])

[[0, 88], [89, 116], [117, 168], [169, 207]]
[[0, 25], [33, 116], [117, 166], [169, 207]]


In [55]:
dependency_spans = []

for i in range(len(data)):
    dependency_spans.append(get_spans(data.iloc[i]["comment"]))

data["dependency_spans"] = dependency_spans

In [57]:
total_count = 0

for i in range(len(data)):
    sentence_count = 0

    for span in data.iloc[i]["span_pairs"]:
        if span in data.iloc[i]["dependency_spans"]:
            sentence_count += 1
    
    total_count += sentence_count/len(data.iloc[i]["span_pairs"])

print(total_count / len(data))

0.48917023270471555


In [93]:
def root(token):
    if token.dep_ in ["ROOT", "rs"]:
        return token
    else:
        return root(token.head)
    
# ROOT 0.48917023270471555
# ROOT & re 0.48731153903567703
# ROOT & cj 0.4750760893002272
# ROOT & dep 0.4880527358975635
# ROOT & rs 0.48964915990778063
# ROOT & dm 0.48821237829858527
# ROOT & par 0.4784741918362608


In [77]:
dependency_spans = []

for i in range(len(data)):
    dependency_spans.append(get_spans(data.iloc[i]["comment"]))

data["dependency_spans"] = dependency_spans

In [78]:
total_count = 0

for i in range(len(data)):
    sentence_count = 0

    for span in data.iloc[i]["span_pairs"]:
        if span in data.iloc[i]["dependency_spans"]:
            sentence_count += 1
    
    total_count += sentence_count/len(data.iloc[i]["span_pairs"])

print(total_count / len(data))

0.48964915990778063
